In [3]:
# Standard library imports
import os
import glob

# Third party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from tensorboard.backend.event_processing import event_accumulator

# Define runs to analyze
run_names = ['run_020']
#run_names = ['run_010', 'run_015']
runs_dir = '../../runs'

all_data = []
for run in run_names:
    run_path = os.path.join(runs_dir, run)
    if not os.path.exists(run_path):
        print(f'Run directory {run_path} does not exist')
        continue
        
    # Load all event files within the run directory (including subdirectories for eval)
    event_files = sorted(glob.glob(os.path.join(run_path, '**', 'events.out.tfevents.*'), recursive=True))
    print(f'Found {len(event_files)} event files for {run}')
    
    for ev_file in event_files:
        # Size guidance to ensure we load enough scalars. 0 means load all.
        size_guidance = {'scalars': 0}
        ea = event_accumulator.EventAccumulator(ev_file, size_guidance=size_guidance)
        ea.Reload()
        
        if 'scalars' in ea.Tags():
            tags = ea.Tags()['scalars']
            for tag in tags:
                for ev in ea.Scalars(tag):
                    all_data.append({'run': run, 'tag': tag, 'step': ev.step, 'value': ev.value})

if all_data:
    df = pd.DataFrame(all_data)
    # Remove duplicates by taking the last value for a given (run, tag, step)
    df = df.groupby(['run', 'tag', 'step']).last().reset_index()
    print('\nData successfully loaded and aggregated.')
else:
    print('\nNo data found in the specified runs.')
    df = pd.DataFrame(columns=['run', 'tag', 'step', 'value'])

Found 2 event files for run_020

Data successfully loaded and aggregated.


In [5]:
# Extract Custom evaluation metrics
eval_tags = [t for t in df['tag'].unique() if 'mAP' in t or 'eval' in t]
eval_tags = [t for t in eval_tags if t not in ['eval/loss']]

if not eval_tags:
    print("No evaluation metrics (mAP) found in the logs.")
else:
    # Compute best metrics per run
    best_metrics = []
    for tag in eval_tags:
        for run in run_names:
            run_data = df[(df['tag'] == tag) & (df['run'] == run)]
            if not run_data.empty:
                max_val = run_data['value'].max()
                best_metrics.append({'Metric': tag, 'Run': run, 'Best Value': max_val})
                
    best_df = pd.DataFrame(best_metrics)

    # Load hierarchy
    # Standard library imports
    import json

    # Third party imports
    import plotly.express as px

    hierarchy_path = '../../conf/hierarchy.json'
    hierarchy = {}
    if os.path.exists(hierarchy_path):
        with open(hierarchy_path, 'r') as f:
            hierarchy = json.load(f)

    groups = {'Global Metrics': [t for t in eval_tags if t.startswith('mAP/') or not t.startswith('mAP_Class/')]} 
    
    for group_name, classes in hierarchy.items():
        # The tags in the logs are like 'mAP_Class/note'
        group_tags = [f'mAP_Class/{cls}' for cls in classes if f'mAP_Class/{cls}' in eval_tags]
        if group_tags:
            groups[group_name] = group_tags
            
    # Ensure any ungrouped mAP_Class tags are not lost
    grouped_tags = set(sum(groups.values(), []))
    ungrouped = [t for t in eval_tags if t not in grouped_tags]
    if ungrouped:
        groups['Other Classes'] = ungrouped

    # Generate tables and plots per group
    for group_name, tags in groups.items():
        group_df = best_df[best_df['Metric'].isin(tags)].copy()
        if group_df.empty:
            continue
            
        # 1. Summary Table
        summary_df = group_df.pivot(index='Metric', columns='Run', values='Best Value')
        for r in run_names:
            if r not in summary_df.columns:
                summary_df[r] = np.nan
                
        def highlight_max(s):
            is_max = s == s.max()
            return ['font-weight: bold; background-color: #d4edda; color: black' if v else '' for v in is_max]
            
        styled_df = (
            summary_df.style
            .apply(highlight_max, axis=1)
            .format("{:.4f}", na_rep="-")
            .set_caption(f"{group_name.replace('_', ' ')} - Maximum Performance by Model")
            .set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'left'), ('font-size', '14px')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px')]},
                {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
            ])
        )
        display(HTML(styled_df.to_html()))
        
        # 2. Plotly Plot
        # Strip the prefix for cleaner legends and ticks
        group_df['Metric Name'] = group_df['Metric'].str.replace('mAP_Class/', '').str.replace('mAP/', '')
        
        fig = px.line(group_df, x='Run', y='Best Value', color='Metric Name', symbol='Metric Name',
                      markers=True, title=f'{group_name.replace("_", " ")} - Plot Comparison',
                      template='plotly_white')
        # Add some padding to Y axis so markers are not cut off
        fig.update_yaxes(range=[max(0, group_df['Best Value'].min()-0.05), min(1.05, group_df['Best Value'].max()+0.05)])
        fig.update_traces(marker=dict(size=12), line=dict(width=2))
        fig.update_layout(xaxis_title='Run', yaxis_title='Best mAP Score', hovermode='x unified')
        fig.show()


Run,run_020
Metric,
mAP/IoU_0.50,0.5118
mAP/IoU_0.50_0.95,0.4192
mAP/IoU_0.75,0.4552


Run,run_020
Metric,
mAP_Class/chord,0.6096
mAP_Class/mRest,0.0006
mAP_Class/note,0.7178
mAP_Class/noteheadBlack,0.9562
mAP_Class/noteheadHalf,0.1943
mAP_Class/noteheadWhole,0.2550
mAP_Class/rest16th,0.7024
mAP_Class/rest8th,0.8388
mAP_Class/restQuarter,0.9276


Run,run_020
Metric,
mAP_Class/barlineDouble,0.0486
mAP_Class/barlineFinal,0.3061
mAP_Class/barlineSingle,0.3149
mAP_Class/clefC,0.9723
mAP_Class/clefF,0.9892
mAP_Class/clefG,0.9839
mAP_Class/fClefChange,0.1089
mAP_Class/keyAccidFlat,0.8664
mAP_Class/keyAccidSharp,0.9032


Run,run_020
Metric,
mAP_Class/beam,0.7537
mAP_Class/dots,0.6981
mAP_Class/flag16thDown,0.0132
mAP_Class/flag8thDown,0.4620
mAP_Class/flag8thUp,0.5046
mAP_Class/slur,0.6108
mAP_Class/stem,0.9608
mAP_Class/tie,0.5602
mAP_Class/tuplet,0.0370


Run,run_020
Metric,
mAP_Class/articStaccatoAbove,0.0585
mAP_Class/fermata,0.6703


...

Run,run_020
Metric,
mAP_Class/dir,0.1758
mAP_Class/dynamicForte,0.8501
mAP_Class/dynamicPiano,0.0010
mAP_Class/pedal,0.7792
mAP_Class/tempo,0.6287


Run,run_020
Metric,
mAP_Class/label,0.2019
mAP_Class/labelAbbr,0.8341
mAP_Class/pgHead,0.9469


Run,run_020
Metric,
mAP_Class/grpSym,0.9396
mAP_Class/repeatMark,0.4356
mAP_Class/voltaBracket,0.3985


Run,run_020
Metric,
mAP_Class/syl,0.5981
mAP_Class/verse,0.5816


Run,run_020
Metric,
mAP_Class/autogenerated,0.8247
mAP_Class/layer,0.5210
mAP_Class/page-margin,0.8733
mAP_Class/system,0.8366


In [6]:
best_df.pivot(index='Metric', columns='Run', values='Best Value').to_markdown()

'| Metric                       |     run_020 |\n|:-----------------------------|------------:|\n| mAP/IoU_0.50                 | 0.511778    |\n| mAP/IoU_0.50_0.95            | 0.419181    |\n| mAP/IoU_0.75                 | 0.455211    |\n| mAP_Class/accidentalFlat     | 0.110996    |\n| mAP_Class/accidentalNatural  | 0.44212     |\n| mAP_Class/accidentalSharp    | 0.55099     |\n| mAP_Class/articStaccatoAbove | 0.0584742   |\n| mAP_Class/autogenerated      | 0.824742    |\n| mAP_Class/barlineDouble      | 0.0485587   |\n| mAP_Class/barlineFinal       | 0.306127    |\n| mAP_Class/barlineSingle      | 0.314852    |\n| mAP_Class/beam               | 0.753739    |\n| mAP_Class/chord              | 0.609646    |\n| mAP_Class/clefC              | 0.972295    |\n| mAP_Class/clefF              | 0.989236    |\n| mAP_Class/clefG              | 0.983858    |\n| mAP_Class/dir                | 0.175837    |\n| mAP_Class/dots               | 0.69812     |\n| mAP_Class/dynamicForte       | 0.8501